In [1]:
import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


### Data Loading Preprocessing and Tokenization

In [3]:
dataset = load_dataset("imdb")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

C:\Users\MSI GF66\miniconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSI GF66\.cache\huggingface\hub\datasets--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [19]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [20]:
example = dataset["train"][0]["text"]
tokens = tokenizer(example, truncation=True, padding="max_length", max_length=256)
print(tokens)

{'input_ids': [101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026, 2678, 3573, 2138, 1997, 2035, 1996, 6704, 2008, 5129, 2009, 2043, 2009, 2001, 2034, 2207, 1999, 3476, 1012, 1045, 2036, 2657, 2008, 2012, 2034, 2009, 2001, 8243, 2011, 1057, 1012, 1055, 1012, 8205, 2065, 2009, 2412, 2699, 2000, 4607, 2023, 2406, 1010, 3568, 2108, 1037, 5470, 1997, 3152, 2641, 1000, 6801, 1000, 1045, 2428, 2018, 2000, 2156, 2023, 2005, 2870, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1996, 5436, 2003, 8857, 2105, 1037, 2402, 4467, 3689, 3076, 2315, 14229, 2040, 4122, 2000, 4553, 2673, 2016, 2064, 2055, 2166, 1012, 1999, 3327, 2016, 4122, 2000, 3579, 2014, 3086, 2015, 2000, 2437, 2070, 4066, 1997, 4516, 2006, 2054, 1996, 2779, 25430, 14728, 2245, 2055, 3056, 2576, 3314, 2107, 2004, 1996, 5148, 2162, 1998, 2679, 3314, 1999, 1996, 2142, 2163, 1012, 1999, 2090, 4851, 8801, 1998, 6623, 7939, 4697, 3619, 1997, 8947, 2055, 2037, 10740, 2006, 4331, 1010, 2016, 2038, 3348, 2007, 2014, 3689, 383

In [21]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

tokenized_dataset = dataset.map(tokenize, batched=True)

In [22]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format(type="torch")

In [36]:
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(10000))
small_test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(2000))

### Model

In [24]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.to(device)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [25]:
for param in model.bert.parameters():
    param.requires_grad = False

In [26]:
for param in model.bert.encoder.layer[-1].parameters():
    param.requires_grad = True

### Fine-Tuning

In [37]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(small_train_dataset, batch_size=8, shuffle=True)
test_dataloader = DataLoader(small_test_dataset, batch_size=8)

In [28]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [38]:
from tqdm import tqdm

model.train()

for epoch in tqdm(range(5)):
    total_loss = 0

    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Loss: {total_loss / len(train_dataloader)}")


 20%|██        | 1/5 [01:50<07:23, 110.84s/it]

Epoch 1, Loss: 0.2832904384605587


 40%|████      | 2/5 [03:44<05:38, 112.73s/it]

Epoch 2, Loss: 0.2522826481565833


 60%|██████    | 3/5 [05:43<03:50, 115.48s/it]

Epoch 3, Loss: 0.21689367880895735


 80%|████████  | 4/5 [07:58<02:03, 123.28s/it]

Epoch 4, Loss: 0.18934139958918095


100%|██████████| 5/5 [10:11<00:00, 122.24s/it]

Epoch 5, Loss: 0.15476424778066575


### Evaluation

In [39]:
from sklearn.metrics import accuracy_score

model.eval()

predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(batch["labels"].cpu().numpy())

accuarcy = accuracy_score(true_labels, predictions)
print(accuarcy)

0.8865


### Inference

In [40]:
def predict_sentiment(text):

    inputs = tokenizer(text, truncation=True, padding="max_length", max_length=256, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    pred = torch.argmax(logits, dim=1).item()
    return "Positive" if pred == 1 else "Negative"

In [41]:
examples = [
    "I absolutely loved this movie! The acting was fantastic.",
    "This was a terrible movie. I wasted two hours of my life.",
    "The plot was okay, but the characters were boring.",
]

for text in examples:
    print(f"Review: {text}")
    print(f"Prediction: {predict_sentiment(text)}\n")

Review: I absolutely loved this movie! The acting was fantastic.
Prediction: Positive

Review: This was a terrible movie. I wasted two hours of my life.
Prediction: Negative

Review: The plot was okay, but the characters were boring.
Prediction: Negative

